In [1]:
!pip install ramantune


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
from ramantune.pipeline.raman_pipeline import RamanPipeline

from ramantune.search.search_space import DenoiserSpace, BaselineSpace, NormalizerSpace, ClassifierSpace, FeatureSelectionSpace
from ramantune.utils.config import DENOISING_STR, BASELINE_STR, NORMALIZE_STR, FEATURE_SELECTION_STR, CLASSIFIER_STR
from ramantune.search.strategies import GridSearchStrategy
from ramantune.search import RamanSearch

from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("bin/ovarian_small.csv")
groups = df['patient']

y = df['label'].values
X = df.drop(columns=['label', 'patient'])

In [6]:
from ramantune.utils import register_algorithm
from ramantune.custom import RamanPipelineStep
from orpl.baseline_removal import bubblefill

@register_algorithm(category=NORMALIZE_STR, name="snv")
class SNVNormalization(RamanPipelineStep):
    def __init__(self):
        super().__init__(self.snv_normalization)

    @staticmethod
    def snv_normalization(spectral_data, spectral_axis):
        mean, std = spectral_data.mean(), spectral_data.std()
        return (spectral_data - mean) / std, spectral_axis

@register_algorithm(category=BASELINE_STR, name="bubblefill")
class BubbleFill(RamanPipelineStep):
    def __init__(self, *, min_bubble_widths=50, fit_order=1):
        super().__init__(
            self._bubblefill_call,
            min_bubble_widths=min_bubble_widths,
            fit_order=fit_order
        )

    @staticmethod
    def _bubblefill_call(spectral_data, spectral_axis, *args, **kwargs):
        raman, bubblefill_b = bubblefill(
            spectral_data,
            min_bubble_widths=kwargs.get("min_bubble_widths", 50),
            fit_order=kwargs.get("fit_order", 1))
        return raman, spectral_axis

In [9]:
def setup_param_grid():
  denoiser_list = [
        DenoiserSpace("savgol", {"window_length": [7], "polyorder": [3]}),
  ]

  baseline_list = [
      BaselineSpace("asls", {"lam": [100]}),
      BaselineSpace("bubblefill", {"min_bubble_widths": [50]}),
  ]

  normalization_list = [
      NormalizerSpace("snv"),
      NormalizerSpace("vector"),
  ]

  feature_selection_list = [
      FeatureSelectionSpace(None), # No feature selection
      FeatureSelectionSpace(PCA(),{"n_components": [0.90, 10]}),
  ]

  classifier_list = [
      ClassifierSpace(SVC(),{"C": [0.1, 10], "kernel": ["rbf"], "gamma": ["scale"]}),
      ClassifierSpace(RandomForestClassifier(),{"n_estimators": [100, 200]})
  ]

  param_list = {
      DENOISING_STR: denoiser_list,
      BASELINE_STR: baseline_list,
      NORMALIZE_STR: normalization_list,
      FEATURE_SELECTION_STR: feature_selection_list,
      CLASSIFIER_STR: classifier_list
  }

  return param_list

In [10]:
estimator = RamanPipeline()
param_grid = setup_param_grid()

In [11]:
search = RamanSearch(estimator=estimator,
                     research_strategy=GridSearchStrategy(),
                     param_grid=param_grid,
                     cv=StratifiedGroupKFold(n_splits=2, random_state=42, shuffle=True),
                     return_train_score=True,
                     n_jobs=1,
                     verbose=10,
                     refit="accuracy")

res = search.fit(X, y, groups=groups)

Fitting 2 folds for each of 48 candidates, totalling 96 fits
[CV 1/2; 1/48] START baseline__algorithm=asls, baseline__lam=100, classifier__C=0.1, classifier__algorithm=SVC(), denoising__algorithm=savgol, denoising__polyorder=3, denoising__window_length=7, feature__algorithm=None, normalize__algorithm=snv
[CV 1/2; 1/48] END baseline__algorithm=asls, baseline__lam=100, classifier__C=0.1, classifier__algorithm=SVC(), denoising__algorithm=savgol, denoising__polyorder=3, denoising__window_length=7, feature__algorithm=None, normalize__algorithm=snv; accuracy: (train=0.667, test=0.333) f1: (train=0.400, test=0.250) patient_accuracy: (train=0.667, test=0.333) precision: (train=0.333, test=0.167) recall: (train=0.500, test=0.500) sensitivity: (train=0.500, test=0.500) specificity: (train=0.500, test=0.500) total time=   0.2s
[CV 2/2; 1/48] START baseline__algorithm=asls, baseline__lam=100, classifier__C=0.1, classifier__algorithm=SVC(), denoising__algorithm=savgol, denoising__polyorder=3, denoi

In [12]:
print(search.get_best_params())
print(search.get_best_score())

{'baseline__algorithm': 'asls', 'baseline__lam': 100, 'classifier__C': 10, 'classifier__algorithm': SVC(), 'denoising__algorithm': 'savgol', 'denoising__polyorder': 3, 'denoising__window_length': 7, 'feature__algorithm': PCA(), 'feature__n_components': 10, 'normalize__algorithm': 'snv'}
0.6190476190476191


In [13]:
result_cv = search.get_cv_results(file_path=f"result_add_preprocessing.csv",
                          return_split_scores=False,
                          return_combined_params=True,
                          round_values=True)

In [14]:
result_cv

,denoising,baseline,normalize,feature,classifier,mean_fit_time,std_fit_time,mean_score_time,std_score_time,split0_test_accuracy,...,std_train_sensitivity,split0_test_patient_accuracy,split1_test_patient_accuracy,mean_test_patient_accuracy,std_test_patient_accuracy,rank_test_patient_accuracy,split0_train_patient_accuracy,split1_train_patient_accuracy,mean_train_patient_accuracy,std_train_patient_accuracy
0,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,None,SVC(C=0.1),0.0835,0.0233,0.1306,0.0364,0.3333,...,0.0,0.3333,0.3333,0.3333,0.0000,25,0.6667,0.6667,0.6667,0.0
1,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,None,SVC(C=10.0),0.0729,0.0017,0.1075,0.0102,0.5714,...,0.0,0.6667,0.3333,0.5000,0.1667,2,1.0000,1.0000,1.0000,0.0
2,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,None,RandomForestClassifier(n_estimators=100.0),0.1627,0.0272,0.1426,0.0400,0.2857,...,0.0,0.3333,0.3333,0.3333,0.0000,25,1.0000,1.0000,1.0000,0.0
3,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,None,RandomForestClassifier(n_estimators=200.0),0.3318,0.0180,0.1586,0.0211,0.3333,...,0.0,0.3333,0.3333,0.3333,0.0000,25,1.0000,1.0000,1.0000,0.0
4,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=0.9),SVC(C=0.1),0.0585,0.0010,0.0840,0.0023,0.3333,...,0.0,0.3333,0.3333,0.3333,0.0000,25,0.6667,0.6667,0.6667,0.0
5,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=10.0),SVC(C=0.1),0.1074,0.0291,0.0846,0.0029,0.3333,...,0.0,0.3333,0.3333,0.3333,0.0000,25,0.6667,0.6667,0.6667,0.0
6,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=0.9),SVC(C=10.0),0.0804,0.0228,0.1189,0.0234,0.6667,...,0.0,0.6667,0.3333,0.5000,0.1667,2,1.0000,1.0000,1.0000,0.0
7,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=10.0),SVC(C=10.0),0.1077,0.0041,0.1492,0.0551,0.7143,...,0.0,0.6667,0.6667,0.6667,0.0000,1,1.0000,1.0000,1.0000,0.0
8,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=0.9),RandomForestClassifier(n_estimators=100.0),0.3302,0.1026,0.2410,0.0774,0.6190,...,0.0,0.6667,0.3333,0.5000,0.1667,2,1.0000,1.0000,1.0000,0.0
9,"savgol(polyorder=3,window_length=7)",asls(lam=100.0),snv,PCA(n_components=10.0),RandomForestClassifier(n_estimators=100.0),0.4399,0.1187,0.3375,0.1031,0.6190,...,0.0,0.6667,0.3333,0.5000,0.1667,2,1.0000,1.0000,1.0000,0.0
